In [1]:
# -*- coding: utf-8 -*-
from turtle import title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os
from ansys.aedt.core import generate_unique_name
from ansys.aedt.core import Desktop, Hfss, constants


In [2]:
def SingleFrequency(
    freq,
    maxPasses,
    maxDeltaS= 0.02,
    name="MySetup"
):
    setup.props["Frequency"] = freq
    setup.props["MaximumPasses"] = maxPasses
    setup.props["MaxDeltaS"] =maxDeltaS
    setup.props["UseIterativeSolver"] = True
    setup.props["SaveRadFieldsOnly"] = True


In [3]:
def LinearCountSweep(
    fStart,
    fStop,
    freqPonts=401,
    units="GHz",
    saveFields=True,
    saveRadFields=True
):
    sweep = hfss.create_linear_count_sweep(
        setup=setup.name,
        units=units,
        start_frequency=fStart,
        stop_frequency=fStop,
        num_of_freq_points = freqPonts,
        name= "sweep",
        sweep_type="Interpolating",
        save_fields=saveFields,
        save_rad_fields=saveRadFields
    )


In [ ]:

def SaveAnalyseSetup(
    nameFolder= "WaveGuide.aedt",
    setup="MySetup"
):
    hfss.save_project(os.path.join(temp_folder, nameFolder))
    hfss.analyze_setup(setup)

In [5]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "False"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)

a=7.11
b=3.56
c=50
t=1

hfss = Hfss(solution_type="Modal")
hfss.modeler.model_units = "mm"

p = hfss.modeler.primitives

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_7467e588-c12f-4cc7-9bc6-e37d609c66ac.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 51085.
PyAEDT INFO: Electronics Desktop started on gRPC port: 51085 after 19.14124870300293 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 15272.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT

In [6]:
outer_WG = p.create_box(
    [-(a + 5)/2 ,-c ,-(b + 5)/2],
    [(a + 5),c,(b + 5)],
    material="copper",
    name="outer_WG"
)

inner_WG_cut = p.create_box(
    [-a/2 ,-c ,-b/2],
    [a,c,b],
    name="inner_WG_cut"
)

p.subtract(
    outer_WG, 
    inner_WG_cut
)

inner_WG = p.create_box(
    [-a/2 ,-c ,-b/2],
    [a,c,b],
    name="inner_WG"
)

port1 = p.create_rectangle(
    constants.AXIS.Y,
    origin=(-a/2,-c,1.78) ,
    sizes=(-b, a),
    name="port1"
)

port2 = p.create_rectangle(
    constants.AXIS.Y,
    origin=(-a/2,0,1.78) ,
    sizes=(-b, a),
    name="port2"
)

PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec


In [7]:

hfss.wave_port(
    assignment=port1.name,                 
    reference=port1,
    integration_line=hfss.AxisDir.ZNeg,    
    impedance=50,
    name=generate_unique_name("port1"),
    renormalize=False,
    terminals_rename=False
)

hfss.wave_port(
    assignment=port2.name,                 
    reference=port2,
    integration_line=hfss.AxisDir.ZNeg,    
    impedance=50,
    name=generate_unique_name("port2"),
    renormalize=False,
    terminals_rename=False
)

PyAEDT INFO: Boundary Wave Port port1_092H88 has been created.
PyAEDT INFO: Boundary Wave Port port2_HLLZ6E has been created.


In [8]:
my_plot =hfss.plot(show=False, plot_air_objects=False)
my_plot.show_axes = False
my_plot.show_grid = False
my_plot.isometric_view = True
my_plot.plot(
    os.path.join(hfss.working_directory, "WaveGuide.jpg")    
)

setup = hfss.create_setup("MySetup")


PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec


Widget(value='<iframe src="http://localhost:51123/index.html?ui=P_0x18a7dd02a70_0&reconnect=auto" class="pyvis…

In [9]:
SingleFrequency("42GHz", 6, 0.02) 

In [10]:
LinearCountSweep(21.0,43.0)

PyAEDT INFO: Linear count sweep sweep has been correctly created.


In [11]:
SaveAnalyseSetup("WaveGuide.aedt")

PyAEDT INFO: Project WaveGuide Saved correctly
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/HFSS correctly changed.
PyAEDT INFO: Solving design setup MySetup
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/HFSS correctly changed.
PyAEDT INFO: Design setup MySetup solved correctly in 0.0h 1.0m 38.0s


In [12]:
d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True